In [ ]:
"""
<시각화 및 특성(feature) 엔지니어링>
 0. 용어정의 : 컬럼=항목=특성=퓨처(feature) 모두 같은 의미
 1. 단일 변수(수치형/범주형) 분포 시각화 및 특성 파악
 2. 다변량 상관관계 분석 및 통계적 유의성(p-value) 검증
 3. 분석 목적에 따른 범주형 데이터 인코딩 (Label, One-Hot)
 4. 수치형 데이터 스케일링 (Standard, MinMax, Robust)
 5. 비즈니스 로직 기반 파생변수(새로운 특성(feature)) 생성
 6. 타겟(target=종속=정답) 변수(Churn 고객이탈 여부 특성) 클래스 불균형 확인 및 시각화
 7. SMOTE 기법을 활용한 오버샘플링 및 불균형 해소
   - K-최근접 이웃(KNN) 알고리즘 원리를 기반으로
     소수(작은 데이터) 클래스(이탈 고객)의 데이터를 가상으로 증식시켜
     클래스(target) 비율을 1:1로 맞추기
 8. 모델 학습을 위한 훈련(학습용)/검증(튜닝용)/테스트(최종 평가용) 데이터 = 6:2:2 분리
"""


### 1. 단일 변수(수치형/범주형) 분포 시각화 및 특성 파악

##### 사용할 라이브러리 불러들이기


In [ ]:
%run ../common/libraries.py

# 이번 학습에서 사용하는 주요 라이브러리
# - pandas(pd) : CSV 및 DataFrame 데이터 처리
# - numpy(np) : 배열 및 수치 데이터 처리
# - matplotlib(plt) : 그래프 시각화
# - seaborn(sns) : 통계 그래프 및 히트맵 시각화
# - scipy.stats : 통계 검정
# - train_test_split : 훈련/검증/테스트 데이터 분리
# - LabelEncoder : 범주형 데이터를 숫자로 변환
# - StandardScaler : 표준화 스케일링
# - MinMaxScaler : 최소/최대 기준 스케일링
# - RobustScaler : 중앙값과 IQR 기준 스케일링
# - SMOTE : 불균형 데이터 오버샘플링


##### 사용할 데이터셋 읽어들이기(전처리 완료 파일)

In [ ]:
### 데이터셋 읽어들이고, 변수명은 df로 받아오기
file_path = "../data/ecommerce_customers_clean.csv"
df = pd.read_csv(file_path)
df.info()
df.head(1)

##### 수치형 데이터(나이, 가입기간)에 대한 분포 확인하기(시각화-히스토그램)

In [1]:
### 하나의 그래프에 여러 그래프를 그리기
# - 하나의 그래프 내에 공간(행렬) 만들기
fig, axes = plt.subplots(1, 2)

### - 첫번째 axes[0]번째에 고객 나이(Age) 분포를 히스토그램으로 시각화
#   -> kde=True 지정 : 커널 밀도 추정 곡선을 함께 그리기
#   -> bins : 막대의 너비 사이즈 조정
sns.histplot(df["Age"], bins=30, kde=True, ax=axes[0], color="skyblue")

# 첫번째 그래프의 제목 넣기
axes[0].set_title("고객 나이(Age) 분포")

### 두번째 axes[1]번째에 서비스 가입 기간(Tenure) 분포를 시각화 하기
sns.histplot(df["Tenure"], bins=24, kde=True, ax=axes[1], color="salmon")

# 두번째 그래프의 제목 넣기
axes[1].set_title("가입 기간(Tenure_월) 분포")

### 서브플롯(여러개의 그래프를 하나의 그래프로)으로 시각화 시에는
# - 항상 마지막에 레이아웃 자동 조정 함수를 넣어줍니다.
# - 행이 많아지는 경우 위/아래 행의 그래프가 겹쳐서 보여지게되어 이를 자동 조종해야함
plt.tight_layout()

# 시각화 그래프 보여주고, 시각화에 사용된 메모리 삭제하기(초기화)
plt.show()

NameError: name 'plt' is not defined

##### 범주형 데이터(성별)의 분포 확인하기(시각화 - 막대그래프-countplot 사용)

In [ ]:
plt.title("고객 성별(Gender) 분포")
# - hue : count에 사용할 컬럼 지정
# - palette : 막대 색상 설정
# - legend : 범례 표시 설정
sns.countplot(data=df, x="Gender", hue="Gender", palette="Set2", legend=False)
plt.show()

### 범주 데이터에 대한 확인은 편차가 있는지 확인...(편향(bias) 유무를 확인)

### 2. 다변량 상관관계 분석 및 통계적 유의성(p-value) 검증

In [ ]:
"""
두 개 이상의 수치형 변수(컬럼, 항목, 특성(feature)) 간의 선형적 연관성(상관계수)을
 - 수치 및 히트맵으로 시각화하고,
 - 통계적 가설 검정(p-value)을 통해
    => 이 변수 간의 상관관계가 우연인지, 우연이 아닌지(유의미한지) 판단함
    => 유의미한 변수들을 찾는 과정임. 즉 모델 훈련에 사용할 특성을 찾는 과정
"""


##### 사용할 라이브러리 불러들이기

##### Pandas의 데이터프레임을 이용한 전체 수치형 변수 간의 상관행렬표(Correlation Matrix) 확인하기

In [ ]:
### 데이터프레임에서 수치형(number) 데이터만 먼저 추출하기(범주형 데이터는 제외)
number_df = df.select_dtypes(include=["number"])
number_df

### 상관행렬표 확인하기 -> 피어슨(pearson) 방식 사용
# - corr() : 상관행렬표(Correlation Matrix) 함수
corr_matrix = number_df.corr(method="pearson")
corr_matrix

### 수치값 해석 방법
#  값의 범위 : 1 ~ -1
#   1 : 완벽한 [양의 상관관계]=우상향=비례
#   0 : 상관 없음
#  -1 : 완벽한 [음의 상관관계]=우하향=반비례

#  *** 절대값으로 해석하면 됨

##### 상관행렬표를 시각화하기(상관행렬도 그리기 - 히트맵 그래프 사용)

In [ ]:
### 그래프 가로/세로 사이즈 조정
plt.figure(figsize=(10, 8))

### 그래프 제목 넣기
plt.title("수치형 특성 간 피어슨 상관계수 히트맵 시각화")

### 히트맵(heatmap) 그래프 그리기
#  - annot=True: 박스 안에 수치 표시,
#  - fmt='.2f': 소수점 둘째 자리까지 표시,
#  - cmap='coolwarm': 한랭/온난 색상 맵 적용
#  - vmin/vmax : 히트맵의 색상 막대(Colorbar)가 표현할 최소값과 최대값을 고정하는 값
sns.heatmap(corr_matrix, annot=True, fmt=".2f", 
            cmap="coolwarm", vmin=-1, vmax=1)

plt.show()

##### 특정 변수간에 상관관계에 대한 유의미성(p-value) 검증

In [ ]:
"""
<유의미성(p-value) 검증 방법>
 - 피어슨 검증 방법(가장 많이 사용되고 있는 방법)
 - 스피어만 검증 방법
"""

In [ ]:
### 피어슨 방식
# - 가입기간(Tenure)과 총결제액(TotalCharges) 강의
#   피어슨 상관관계 계수와 유의미 계수(p-value) 계산하기
# - 유의미 계수(p-value)이 0.05보다 작다면 유의미하다고 판단
#                                   크면 유의미 하지 않다고 판단(우연의 일치)
# - 수식으로 표현하여 해석 : p-value < 0.05 이므로 유의미함
pearson_coef, p_value_pearson = stats.pearsonr(df["Tenure"], df["TotalCharges"])
print(f"가입기간 vs 총결재액 상관관계 계수 = {pearson_coef:.3f} / p-value = {p_value_pearson}")

### p-value의 값이 0.05 미만이면 -> "가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 합니다" 출력
#   p-value의 값이 0.05 이상이면 -> "가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 하지 않습니다." 출력
if p_value_pearson < 0.05 :
    print("가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 합니다")
else :
    print("가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 하지 않습니다." )

In [ ]:
### 스피어만 방식
# - 리뷰점수(ReviewScore)과 월결제액(MonthlyCharges) 강의
#   스피어만 상관관계 계수와 유의미 계수(p-value) 계산하기
# - 유의미 계수(p-value)이 0.05보다 작다면 유의미하다고 판단
#                                   크면 유의미 하지 않다고 판단(우연의 일치)
# - 수식으로 표현하여 해석 : p-value < 0.05 이므로 유의미함
spearman_coef, p_values_pearman = stats.spearmanr(df["ReviewScore"], df["MonthlyCharges"])
print(f"가입기간 vs 총결재액 상관관계 계수 = {spearman_coef:.3f} / p-value = {p_values_pearman}")

### p-value의 값이 0.05 미만이면 -> "가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 합니다" 출력
#   p-value의 값이 0.05 이상이면 -> "가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 하지 않습니다." 출력
if p_values_pearman < 0.05 :
    print("가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 합니다")
else :
    print("가입기간(Tenure)과 총결제액(TotalCharges) 간의 관계는 유의미 하지 않습니다." )

In [ ]:
"""
Churn 컬럼과 아래 6개의 컬럼들과 각각 피어슨 방식으로 p-value값을 확인하여 
 -> 유의미한지 안한지 각각 출력해 주세요.

    "Age": 고객 나이 (int)
    "Tenure": 가입 기간_월단위 (int)
    "MonthlyCharges": 월 평균 결제액 (float)
    "TotalCharges": 총 결제액 (float)
    "SupportCalls": 고객센터 문의 횟수 (int)
    "ReviewScore": 리뷰 평균 점수 (float)
"""
### 반복을 위해서 사용할 컬럼명 6개를 리스트 타입으로 정의하기
cols = df.columns.drop(['CustomerID', 'Gender', 'Churn'])

### 반복하기
for col in cols:
    # print(col)
    pearson_coef, p_value_pearson = stats.pearsonr(df["Churn"], df[col])
    print(f"Churn vs {col} 상관관계 계수 = {pearson_coef:.3f} / p-value = {p_value_pearson:.3f}")

    if p_value_pearson < 0.05 :
        print(f"Churn vs {col} 간의 관계는 유의미 합니다 \n")
    else :
        print(f"Churn vs {col} 간의 관계는 유의미 하지 않습니다. \n")

### 3. 분석 목적에 따른 범주형 데이터 인코딩 (Label, One-Hot)

In [ ]:
"""
- 머신러닝 알고리즘은 텍스트를 연산할 수 없으므로,
  -> 문자형 데이터를 숫자로 변환해야 함

- 데이터의 성격(서열 유무)에 따라 라벨 인코딩과 원-핫 인코딩으로 구분하여 적용
  -> 라벨 인코딩 예시 : 연령대 10, 20, 30 / 종류 0, 1, 2, 3 처럼 데이터 변환
  -> 원-핫 인코딩 예시 : 성별 컬럼(특성)을 남자 컬럼(특성)과 여자 컬럼(특성)으로 추가하여
                     : 남자가 있는 행(row)의 경우에는
                        > 남자 컬럼(특성)에는 1로, 여자 컬럼(특성)에는 0으러 입력하여 데이터 변환
                     : 컬럼을 추가하여 학습에 유리하도록 처리하는 방법
"""

##### 사용할 라이브러리 불러들이기

In [ ]:
### 사용할 데이터를 복제하여 사용(원본 훼손을 막기 위해)
# - 복제의 의미 : 새로운 데이터프레임 메모리를 생성함
df_encoded = df.copy()
df_encoded.info()
df_encoded.head(1)

##### 라벨 인코딩(label Encoding) 처리하기 - Gender 데이터 처리

In [ ]:
### 라벨인코딩 클래스 객체 사용해야함
# - 라벨인코딩 클래스 생성하기
le_gender = LabelEncoder()
le_gender

### Gender 성별 데이터의 범주에 대한 숫자로 변경하기 위한 패턴 찾기
# - 범주값을 스스로 찾아서 숫자로 변경하게끔 패턴 학습 시킴
# - fit : 패턴만 찾기
# - transform : 찾은 패턴으로 숫자로 변경하기
# - fit_transform : 패턴을 찾아서 동시에 숫자로 변경
df_encoded["Gender_Label"] = le_gender.fit_transform(df_encoded["Gender"])

### 라벨인코더가 찾아낸 범주들 추출
print(le_gender.classes_)

df_encoded.head(3)

##### 원-핫 인코딩(One-Hot Encoding) 하기

In [ ]:
### 원핫 인코딩을 위한 컬럼은 임의로 샘플 컬럼을 만들어서 해보겠습니다.

### 멤버십(Membership)이라는 임의 샘플 컬럼 생성하기(데이터는 렌덤으로 생성)
# - len(df_encoded) 데이터프레임의 전체 행의 갯수만큼
# - 랜덤하게 값을 선택해서 처리
df_encoded["Membership"] = np.random.choice(["Basic", "Silver", "Gold"], size=len(df_encoded))
df_encoded.head(3)

In [ ]:
### 원-핫 인코딩 처리하기 : 판다스 라이브러리의 get_dummies() 함수사용
# - Membership 원본 컬럼은 삭제되고, 범주값들이 컬럼으로 만들어짐
# - 컬럼명 이름 규칙 : 원본컬럼명_범주명
df_encoded = pd.get_dummies(df_encoded, columns=["Membership"], dtype=int)

# print(df_encoded["Membership"])
df_encoded.info()
df_encoded.head(3)

### 4. 수치형 데이터 스케일링 (Standard, MinMax, Robust)

In [ ]:
"""
<데이터 스케일링>
 - 서로 다른 단위와 범위를 가진 수치형 변수들(예: 10~70세 나이 vs 10000달러 결제액)을
   -> 모델이 동일한 비중으로 학습할 수 있도록 정규화/표준화하는 방법을 "데이터 스케일링" 이라고 칭함

** 표준화(Standardization)와 정규화(Normalization)
 - StandardScaler : 표준화(Standardization)
                  : 데이터의 분포(Distribution)를 중심성 위주로 정렬함
                  : 데이터를 평균이 0, 표준편차가 1인 고르게 분포된 형태로 변환함
                  : 음수(-)의 값이 나올수도 있음
                  : 이상치에 비교적 덜 민감함. (평균과 분산이 이동할 뿐 데이터 균형이 깨지지 않음)
                  : 현업에서 가장 많이 사용함(70%)
                    (보통 전처리가 선행되기에 대부분의 머신러닝 알고리즘과 수학적으로 가장 잘 맞고 안정적임)

 - MinMaxScaler   : 정규화(Normalization)
                  : 데이터의 범위(Scale)를 제한함
                  : 데이터의 최솟값을 0, 최댓값을 1로 고정하여 [0, 1] 범위로 압축함
                  : 이상치에 매우 취약함. (특이값 하나 때문에 정상 데이터들이 좁은 구역으로 뭉침)
                  : 현업에서 두번째로 가끔 사용함(20%)
                    (이미지 데이터(0~255) 처리나 데이터의 최댓값/최솟값이 명확히 정해져 있을 때 사용함)
                   
 - RobustScaler   : 표준화의 변형 (이상치 최소화 방법)  
                  : 평균과 표준편차 대신 중앙값(Median)과 IQR(사분위수 범위)을 사용하여
                    -> 이상치(Outlier)의 영향을 최소화하는 표준화 기법
                  : 음수(-)의 값이 나올수도 있음
                  : 데이터에 말도 안 되게 크거나 작은 이상치(Outlier)가 불규칙하게 섞여 있을 때,
                    -> Standard나 MinMax를 쓰면 모델이 망가지므로
                    => RobustScaler를 최우선으로 선택
                  : 현업에서 드물게 사용함(10% 미만)
                    (금융 사기 탐지(Fraud) 또는 이상 거래 등 극단적인 아웃라이어(이상치 분석)가
                     핵심 데이터로 분석될 때 사용함)
"""

In [ ]:
### 스케일링할 데이터(특성)을 별도로 추출하기
# - 보통 연속형 데이터를 스케일링 처리함
scale_cols = ["Age", "Tenure", "MonthlyCharges", "TotalCharges"]
scale_cols

In [ ]:
df_encoded[["Age", "Tenure", "MonthlyCharges", "TotalCharges"]]
df_encoded[scale_cols]

In [ ]:
### StandardScaler 사용하기.
# 클래스 생성하기
std_scaler = StandardScaler()
std_scaler

# 특성들에 대해서 변경할 패턴 찾고 값 생성하기
print(len(df_encoded))
std_scaler.fit_transform(df_encoded[scale_cols])

### 스케일링한 4개 컬럼의 값에 대하여 데이터프레임에 새로운 컬럼 추가하여 값 저장시키기
# - 새로운 컬럼 이름 규칙 : "Age_std", "Tenure_std", "MonthlyCharges_std", "TotalCharges_std"
[col + "_std" for col in scale_cols]
# df_encoded[["Age_std", "Tenure_std", "MonthlyCharges_std", "TotalCharges_std"]] = std_scaler.fit_transform(df_encoded[scale_cols])
df_encoded[[col + "_std" for col in scale_cols]] = std_scaler.fit_transform(df_encoded[scale_cols])
df_encoded.head(1)

In [ ]:
### MinMaxScaler 사용하기.
# 클래스 생성하기
mn_scaler = MinMaxScaler()
mn_scaler

# 특성들에 대해서 변경할 패턴 찾고 값 생성하기
print(len(df_encoded))
mn_scaler.fit_transform(df_encoded[scale_cols])

### 스케일링한 4개 컬럼의 값에 대하여 데이터프레임에 새로운 컬럼 추가하여 값 저장시키기
# - 새로운 컬럼 이름 규칙 : "Age_mn", "Tenure_mn", "MonthlyCharges_mn", "TotalCharges_mn"
df_encoded[[col + "_mn" for col in scale_cols]] = mn_scaler.fit_transform(df_encoded[scale_cols])
df_encoded.head(1)

In [ ]:
### RobustScaler 사용하기.
# 클래스 생성하기
rb_scaler = RobustScaler()
rb_scaler

# 특성들에 대해서 변경할 패턴 찾고 값 생성하기
print(len(df_encoded))
rb_scaler.fit_transform(df_encoded[scale_cols])

### 스케일링한 4개 컬럼의 값에 대하여 데이터프레임에 새로운 컬럼 추가하여 값 저장시키기
# - 새로운 컬럼 이름 규칙 : "Age_rb", "Tenure_rb", "MonthlyCharges_rb", "TotalCharges_rb"
df_encoded[[col + "_rb" for col in scale_cols]] = rb_scaler.fit_transform(df_encoded[scale_cols])
df_encoded.head(1)

In [ ]:
"""
위의 3개의 스케일링 클래스를 한번에 실행해서 처리될 수 있도록 처리해 주세요..
 - 컬럼명 규칙 : _std2, _mn2, _rb2
"""
df_encoded2 = df_encoded.copy()

dict_ = {"_std2" : StandardScaler(), "_mn2" : MinMaxScaler(), "_rb2" : RobustScaler()}

for c_nm, s in dict_.items():
    df_encoded2[[col + c_nm for col in scale_cols]] = s.fit_transform(df_encoded2[scale_cols])

df_encoded2.head(3)


### 5. 비즈니스 로직 기반 파생변수(새로운 특성(feature)) 생성

In [ ]:
"""
<파생 변수(새로운 특성 추가)>
 - 단순 데이터 제공을 넘어
 - 비즈니스적 통찰력을 바탕으로 모델의 성능을 끌어올릴 수 있는
 - 새로운 특징(Feature)을 생성(보통 책에서는 "창조"라는 단어를 사용하기도 함)
"""

##### 월 평균 기여도 특성(컬럼) 생성(추가)하기

In [ ]:
### 월평균기여도 = 총결제금액(TotalCharges) / 가입개월수(Tenure)
# - 고객의 실질적인 월평균기여도를 의미함
# - 분모가 0인 경우를 방지하기 위해서 -> 매우 작은 값을 더해주기(0.00001 = 1e-5)
# df_encoded.info()

df_encoded["AvgPaymentPerMonth"] = df_encoded["TotalCharges"] / (df_encoded["Tenure"] + 1e-5)
df_encoded.head(1)

##### 연령대 특성(컬럼) 추가하기

In [ ]:
### 연령대(Age_Group) = 연령(Age) 특성의 값을 이용하여 구간 나누기 함수 사용하여 만들기
# - cut() 함수 : pandas 라이브러리에서 제공하는 구간나누기 함수
# - 연령대는 범주형 데이터로 생성됨

### 처리 순서
# 1. 범주 구간의 조건을 리스트로 정의 -> cut 함수의 속성으로 사용
bins = [0, 19, 29, 39, 49, 59, 100]

# 2. 실제 범주값으로 사용할 리스트 정의 -> cut 함수의 속성으로 사용
labels = ["10s", "20s", "30s", "40s", "50s", "60s+"]

# 3. 원본 Age 값을 이용하여 cut() 함수 사용
df_encoded["Age_Group"] = pd.cut(df_encoded["Age"], bins=bins, labels=labels)
df_encoded.head(3)

##### 생성된 연령대 데이터를 다시 숫자 인덱스로 변환(code 속성 사용)

In [ ]:
### 숫자 인덱스로 변환 : codes
# ["10s", "20s", "30s", "40s", "50s", "60s+"] =>  0, 1, 2, 3, 4, 5
df_encoded["Age_Group_Code"] = df_encoded["Age_Group"].cat.codes
df_encoded.head(3)

### 6. 타겟(target=종속=정답) 변수(Churn 고객이탈 여부 특성) 클래스(범주값, 0/1) 불균형 확인 및 시각화

In [ ]:
"""
 - 독립변수 : 문제들(여러개의 특성을 의미함-2차원)
            : 주로 사용되는 변수명 -> X, input
            
 - 종속변수 : 정답(한개 특성을 의미함-1차원) 
            : 주로 사용되는 변수명 -> y, target
"""

In [ ]:
"""
예측하고자 하는 종속변수(Churn-고객 이탈여부) 데이터에 대한
  - '이탈(1)' 클래스의 데이터 수가 -> '유지(0)' 클래스에 비해
  - 얼마나 적은지 파악하여,
  - 훈련 중에 과적합 방지 대책(SMOTE)을 세움

<과적합>
 - 과대적합 : 모델 훈련 결과가 검증 결과보다 너무 높은 결과를 나타내는 경우
            : 모델 훈련 결과가 1이 나오는 경우
 - 과소적합 : 모델 훈련 결과가 검증 결과보다 너무 낮은 결과를 나타내는 경우
"""


In [ ]:
### Churn(고객 이탈여부) 특성의 범주별 갯수 확인하여 불균형 확인하기
# - value_counts() 함수 사용
churn_counts = df_encoded["Churn"].value_counts()
churn_counts

In [ ]:
### 원형 차트(Pie Chart)를 통해
#  - 불균형 상태를 시각화함

# 그래프 전체 너비와 높이 사이즈 지정
plt.figure(figsize=(6, 6))

# pie 함수로 차트를 그리며,
#  - labels : 각 파이 조각에 표시될 이름(라벨)을 작성
#           : 첫 번째 조각에는 '유지 고객(0)',
#           : 두 번째 조각에는 '이탈 고객(1)'이라는 설명 표시
#  - autopct : 각 파이 조각이 전체에서 차지하는 비율(%)을 자동으로 계산하여
#              차트 위에 텍스트로 표시해 주는 기능
#            : '%1.1f%%' 형식을 사용하면 -> 소수점 아래 첫째 자리까지(예: 34.5%)
#  - colors : 차트 조각들의 색상을 지정하는 속성
#           : 순서대로 첫 번째 라벨(Retained)은 연한 파란색(##66b3ff),
#                     두 번째 라벨(Churned)은 경고나 주의를 뜻하는 연한 빨간색/핑크색(##ff9999)으로 채색
#  - startangle : 차트가 그려지기 시작하는 첫 조각의 각도(위치)를 지정
#               : 기본값(0)은 3시 방향에서 시작하지만,
#               : 90을 주면 12시 정각 방향에서 시작하여 -> 시계 반대 방향으로 조각들을 배치
plt.pie(churn_counts,
        labels=['Retained(유지 고객(0))', 'Churned(이탈 고객(1))'],
        autopct='%1.1f%%',
        colors=['#66b3ff', '#ff9999'],
        startangle=90)

# 어떤 모니터 환경에서도 파이 차트가 찌그러진 타원이 되지 않고
#   - 완벽한 원형으로 고정되어 출력되도록 함
plt.axis('equal')

# 차트 제목을 설정함
plt.title("타겟 변수(Churn) 클래스 비율")

# 그래프를 출력하고 시각화 메모리 초기화함
plt.show()


"""
<시각화 해석>
 - 파이 차트를 통해 이탈(1) 고객의 비율이 전체의 약 40% 수준으로 불균형은 다소 크지는 않은 데이터임
 - 사실상 6(유지) : 4(이탈)의 비율은 현업에서 가장 이상적인 데이터 분포에 속함
 - 만약, 유지 고객의 데이터가 이탈 고객의 데이터보다 매우 많은 분포(가령 8:2) 차이(편향)를 보인다면,
         모델은 무조건 "이탈하지 않는다(0)"라는 패턴에 집중되어 이탈에 대한 훈련이 잘 안될 수 있음
         즉, 이탈 고객을 잘 잡아내지 못하는 문제가 발생할 수 있음
         이런 경우, 비슷한 비율로 맞추기 위한 방법을 찾아야함
"""


### 7. SMOTE 기법을 활용한 오버샘플링 및 불균형 해소

In [ ]:
"""
   - K-최근접 이웃(KNN) 알고리즘 원리를 기반으로
     소수(작은 데이터) 클래스(이탈 고객)의 데이터를 가상으로 증식시켜
     클래스(target) 비율을 1:1로 맞추기
"""

In [ ]:
"""
<SMOTE(Synthetic Minority Over-sampling Technique) 기법>
 - 머신러닝에서 데이터의 클래스(종속변수) 불균형(Class Imbalance) 문제를 해결하기 위해 사용하는
  -> 대표적인 오버샘플링(Over-sampling) 기법임
  -> 데이터 불균형이 매우 큰 경우에 주로 사용되는 기업임

 - 예를 들어, 고객 이탈 예측 데이터에서 '유지 고객(0)'은 9,900명인데 '이탈 고객(1)'은 100명뿐이라면,
             모델은 이탈 고객을 맞추는 방법을 제대로 학습하지 못함
             이때 데이터가 적은 쪽(소수 클래스)을 인위적으로 늘려주는 역할을 하는
             기법으로 SMOTE 기법이 주로 사용됨

 - SMOTE 기법은 K-최근접 이웃(KNN) 알고리즘 원리를 기반으로 처리됨
   -> 소수 클래스(이탈 고객)의 데이터를 가상으로 증식시켜 클래스 비율을 1:1로 맞춤

<SMOTE 데이터 합성 과정 : 라이브러리가 알아서 처리하게됨>
 (1)소수 데이터(이탈 고객) A 선택 ─► (2)주변의 가까운 이웃(K-NN) B 선택
    ─► (3)두 데이터 사이의 직선(선분) 확인 ─► (4)그 선 위에 무작위로 새로운 데이터 C 생성

 (1) 소수 클래스(이탈 고객) 데이터 중 하나(A)를 무작위로 선택
 (2) 데이터 A와 가장 가까운 거리에 있는 소수 클래스(이탈 고객) 데이터들(K-최근접 이웃, K-NN)을 찾은 뒤,
     그중 하나(B)를 선택
 (3) 두 데이터 A와 B를 연결하는 직선을 그림
 (4) 이 직선상의 임의의 위치에 새로운 가상의 데이터(C)를 생성하여 추가하게 됨
"""

##### 데이터를 생성하기 위해서 문제(독립변수)와 정답(종속변수)를 구분하기

In [ ]:
### SMOTE에서는 독립변수(문제)를 만들어서 종속변수(정답)을 지정합니다.
# - 따라서 문제와 정답을 분리해서 알려주어야 함.
# - 문제를 이용해서 문제와 가가운 값들의 새로운 문제를 만들어냄

### 독립변수(X)와 종속변수(y) 데이터 분리
# - 독립변수(X) : 문제 데이터 (특성 데이터라고 칭함)
# - 종속변수(y) : 정답 데이터 (타겟 데이터라고 칭함)

### 원본 데이터프레임에서 실제로 문제로 사용할 독립변수들과 종속변수 추출하기
# 독립변수의 특성 : 정답과 사용하지 않는 특성들을 제외하고 추출하기
# 종속변수 : Churn 컬럼의 데이터 사용

### 사용하지 않는 특성 이름을 별도 리스트로 정의해서 사용
cols_to_drop = ["CustomerID", "Age", "Tenure", "MonthlyCharges", 
                "TotalCharges", "Gender", "Churn", "Age_Group"]

### 독립변수 만들기 : 원본 데이터프레임에서 사용하지 않는 특성(컬럼=열)을 삭제
# - axis=1 : 열단위 삭제
X_imbalanced = df_encoded.drop(cols_to_drop, axis=1)
print(X_imbalanced.shape)

### 종속변수 만들기 : 원본 데이터프레임에서 Churn 데이터만 추출
y_imbalanced = df_encoded["Churn"]
y_imbalanced.shape

In [ ]:
X_imbalanced.info()

##### 새로운 샘플 데이터(Over-Sample) 생성하기

In [ ]:
### SMOTE 클래스 생성하기
# - SMOTE 클래스 : 샘플 데이터를 랜덤(원본 데이터에서 벗어나지 않는 랜덤)하게 생성하는 클래스
# - random_state=42 : 랜덤한 규칙(번호)을 지정
#                   : 규칙은 번호로 지정하며, 번호에 의미를 부여하지 않습니다.
#                     (42번 랜덤규칙이라는 의미이지만, 다른 번호 사용해도됨, 주로 42번 사용됨)
smote = SMOTE(random_state=42)
smote

### 새로운 샘플 데이터를 생성하는 함수(기능) 호출하기
X_balanced, y_balanced = smote.fit_resample(X_imbalanced, y_imbalanced)
print(X_balanced.shape, y_balanced.shape)
# X_balanced

### 불균형을 해소되었는지 종속변수의 각 범주 값에 대한 갯수 확인
y_balanced.value_counts()


### 8. 모델 학습을 위한 훈련(학습용)/검증(튜닝용)/테스트(최종 평가용) 데이터 = 6:2:2 분리

In [ ]:
"""
<데이터 분리 기준>
 - 일반화 성능을 극대화하기 위해 전체 데이터를 학습용(Train), 튜닝용 검증(Validation), 최종 평가용(Test)으로 데이터 분리
   
 - 주로 사용하는 분리 비율 : 원본 데이터의 행의 갯수를 기준으로 아래 비율처럼 분리
   => 주로 훈련:검증:테스트 = 6 : 2 : 2
   => 데이터가 매우 많은 경우 훈련:검증:테스트 = 7 : 1.5 : 1.5
   => 데이터가 매우 작은 경우 훈련:검증:테스트 = 5 : 2.5 : 2.5
"""

##### 사용할 라이브러리 불러들이기

##### 데이터 분리하기 (6:2:2)

In [ ]:
### 1차 분할: 전체 데이터를 훈련 데이터(60%)와 임시(검증+테스트) 데이터(40%)로 분리함
#  - test_size=0.4 : 임시 데이터(검증+테스트) 비율을 지정
#  - stratify=y_balanced : 분리 비율에 맞게 종속변수(클래스) 비율도 일정하게 분리 유지
#  - random_state=42 : 데이터 행의 순서를 랜덤한 규칙으로 섞기

### 반환되는 값 순서 : 훈련독립, 테스트독립, 훈련종속, 테스트종속
# - train(60% 데이터) : temp(40% 데이터)
X_train, X_temp, y_train, y_temp = train_test_split(X_balanced, y_balanced, 
                                    test_size=0.4, stratify=y_balanced, random_state=42)

### 받아온 차원 확인 : 행의 갯수와 차원 꼭 확인하기...
print(X_train.shape, y_train.shape)
print(X_temp.shape, y_temp.shape)

In [ ]:
### 2차 분할 : temp 데이터를 5:5로 분할
# 반환되는 값 순서 : 검증독립, 테스트독립, 검증종속, 테스트종속
# - val(50% 데이터) : test(50% 데이터)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, 
                                    test_size=0.5, stratify=y_temp, random_state=42)

### 받아온 차원 확인 : 행의 갯수와 차원 꼭 확인하기...
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
X_train

### train, val, test 데이터를 모두 저장하기

In [ ]:
"""
<저장 방법>
 - 각각의 데이터프레임에 독립변수와 종속변수 저장시켜서 관리
 - train => X_train + y_train 
 - val   => X_val   + y_val 
 - test  => X_test  + y_test 
"""
### 행단위 또는 열단위 데이터 합치는 함수(판다스에서 제공함)
# - 첫번째값 : 리스트 타입으로 합칠 데이터프레임(또는 시리즈)을 순서대로 추가
# - 두번째값 : axis=0(행, 아래로 추가, 디폴트값으로 생략가능), axis=1(열, 옆으로 추가)
df_train = pd.concat([X_train, y_train], axis=1)
df_val   = pd.concat([X_val, y_val],     axis=1)
df_test  = pd.concat([X_test, y_test],   axis=1)


In [ ]:
display(df_train.head(1))
display(df_val.head(1))
display(df_test.head(1))

In [ ]:
### 파일로 저장하기
# - 저장위치 : dataset 폴더
# - df_train -> 파일명 : train_data.csv
# - df_val -> 파일명 : val_data.csv
# - df_test -> 파일명 : test_data.csv
df_train.to_csv("../data/train_data.csv", index=False)
df_val.to_csv("../data/val_data.csv", index=False)
df_test.to_csv("../data/test_data.csv", index=False)

In [ ]:
pd.read_csv("../data/train_data.csv").head()
pd.read_csv("../data/val_data.csv").head()
pd.read_csv("../data/test_data.csv").head()